# Merge Collected News Sources

Combines the SAP News RSS articles and the GNews API articles into a single dataset, builds a unified `text` field for downstream NLP processing, and saves the merged result to `data/Merged_data.json`.

In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.set_option("display.expand_frame_repr", False)

## Load both sources

In [ ]:
DATA_DIR = Path("..") / "data"

sap_df = pd.read_json(DATA_DIR / "sap_news.json")
gnews_df = pd.read_json(DATA_DIR / "gnews_articles.json")

print(f"SAP News: {len(sap_df)} rows | GNews: {len(gnews_df)} rows")

## Align columns and concatenate

Both sources are produced with the same schema (`title`, `description`, `content`, `published`, `source`, `url`), so no renaming is needed before merging — we just reorder the columns for safety and concatenate.

In [ ]:
cols = ["title", "description", "content", "published", "source", "url"]

sap_df = sap_df[cols]
gnews_df = gnews_df[cols]

all_docs = pd.concat([sap_df, gnews_df], ignore_index=True)
all_docs.shape

## Build a combined text field

Missing values are filled with empty strings first, so a `NaN` in any single column doesn't propagate and blank out the whole concatenated `text` field.

In [ ]:
text_cols = ["title", "description", "content"]
all_docs[text_cols] = all_docs[text_cols].fillna("")

all_docs["text"] = (
    all_docs["title"] + " " + all_docs["description"] + " " + all_docs["content"]
)

all_docs["text_length"] = all_docs["text"].str.len()
all_docs["text_length"].describe()

## Save merged dataset

In [ ]:
output_path = DATA_DIR / "Merged_data.json"
all_docs.to_json(output_path, orient="records", indent=4)

print(f"Saved {len(all_docs)} merged documents to {output_path}")